# 💧 LFM2 Inference with Transformers

This notebook demonstrates how to run [LFM2](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda) and [LFM2.5](https://huggingface.co/collections/LiquidAI/lfm25-6839e3e26b2a9fdbde95b341) models using the [Transformers](https://github.com/huggingface/transformers) library.

GPU runtime is recommended for faster inference (`Runtime` → `Change runtime type` → `T4 GPU`).

## Installation

Install the required dependencies:

In [ ]:
!uv pip install "transformers>=5.0.0" "torch==2.9.0" accelerate

## Basic Text Generation

Load the model and generate text using `generate()`:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model_id = "LiquidAI/LFM2.5-1.2B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Generate answer
prompt = "What is C. elegans?"
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

output = model.generate(**inputs, max_new_tokens=512)

# Decode only the newly generated tokens
input_length = inputs["input_ids"].shape[1]
response = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)
print(response)

## Generation Parameters

Control generation behavior with `GenerationConfig`:

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

prompt = "Explain quantum computing in simple terms."
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

output = model.generate(**inputs, generation_config=generation_config)
input_length = inputs["input_ids"].shape[1]
response = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)
print(response)

## Streaming Generation

Stream responses as they're generated:

In [ ]:
from transformers import TextStreamer

prompt = "Tell me a story about space exploration."
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
output = model.generate(**inputs, streamer=streamer, max_new_tokens=512)

## Vision Models

LFM2-VL models support both text and image inputs:

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image

# Load vision model and processor
model_id = "LiquidAI/LFM2.5-VL-1.6B"
vision_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16"
)

# IMPORTANT: tie lm_head to input embeddings (transformers v5 bug)
vision_model.lm_head.weight = vision_model.get_input_embeddings().weight

processor = AutoProcessor.from_pretrained(model_id)

# Load image
url = "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
image = load_image(url)

# Create conversation
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What is in this image?"},
        ],
    },
]

# Generate response
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    tokenize=True,
).to(vision_model.device)

outputs = vision_model.generate(**inputs, do_sample=True, temperature=0.1, min_p=0.15, repetition_penalty=1.05, max_new_tokens=64)
response = processor.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)

## Resources

- [LFM2 Documentation](https://docs.liquid.ai/docs/inference/transformers)
- [LFM2 Models on Hugging Face](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda)
- [Transformers Documentation](https://huggingface.co/docs/transformers)